In [ ]:
from vistiq.segment import MicroSAMSegmenter, MicroSAMSegmenterConfig, SegmentationFlow, SegmentationFlowConfig
from vistiq.io import ImageLoader, ImageLoaderConfig
from vistiq.core import FuncProcessor, FuncProcessorConfig, Tiler, TilerConfig, Untiler, UntilerConfig
from vistiq.preprocess import ResizeConfig, Resize, RescaleConfig, Rescale, DoG, DoGConfig, PreprocessorConfig, Preprocessor
from vistiq.segment import RegionFilterConfig, RegionFilter, RangeFilterConfig, RangeFilter, RegionAnalyzerConfig, RegionAnalyzer 
from vistiq.utils import ArrayIteratorConfig 
from vistiq.analysis import CoincidenceDetectorConfig, CoincidenceDetector
import supervision as sv

from skimage.filters import gaussian
from skimage.exposure import rescale_intensity, adjust_sigmoid, adjust_gamma
import stackview
import os
import numpy as np
from joblib import Parallel, delayed
import math
import logging

In [ ]:
import vistiq
logger = logging.getLogger(vistiq.__name__)

import torch
logger.info(f"Torch version: {torch.__version__}. Cuda enabled: {torch.cuda.is_available()}")

# Functions and Classes

In [ ]:
from typing import Literal

In [ ]:
def box_iou_batch_3d(
    boxes_true: np.typing.NDArray[np.number],
    boxes_detection: np.typing.NDArray[np.number],
    overlap_metric: Literal["IOU", "IOS"] = "IOU"
) -> np.ndarray[np.float32]:
    """
    Adapted for 3d from https://github.com/roboflow/supervision/blob/develop/src/supervision/detection/utils/iou_and_nms.py
    
    Compute pairwise overlap scores between batches of bounding boxes.

    Supports standard IOU (intersection-over-union) and IOS
    (intersection-over-smaller-area) metrics for all `boxes_true` and
    `boxes_detection` pairs. Returns a matrix of overlap values in range
    `[0, 1]`, matching each box from the first batch to each from the second.

    Args:
        boxes_true: Array of reference boxes in
            shape `(N, 4)` as `(x_min, y_min, z_min, x_max, y_max, z_max)`.
        boxes_detection: Array of detected boxes in
            shape `(M, 4)` as `(x_min, y_min, z_min, x_max, y_max, z_min)`.
        overlap_metric: Overlap type.
            Use `OverlapMetric.IOU` for intersection-over-union,
            `OverlapMetric.IOS` for intersection-over-smaller-area.
            Defaults to `OverlapMetric.IOU`.

    Returns:
        Overlap matrix of shape `(N, M)`, where entry
            `[i, j]` is the overlap between `boxes_true[i]` and
            `boxes_detection[j]`.

    Raises:
        ValueError: If `overlap_metric` is not IOU or IOS.

    Examples:
        ```pycon
        >>> import numpy as np
        >>> import supervision as sv
        >>> boxes_true = np.array([
        ...     [100, 100, 200, 200],
        ...     [300, 300, 400, 400]
        ... ])
        >>> boxes_detection = np.array([
        ...     [150, 150, 250, 250],
        ...     [320, 320, 420, 420]
        ... ])
        >>> sv.box_iou_batch_3d(
        ...     boxes_true, boxes_detection, overlap_metric=sv.OverlapMetric.IOU
        ... )
        array([[0.14285..., 0.        ],
               [0.        , 0.47058...]], dtype=float32)
        >>> sv.box_iou_batch(
        ...     boxes_true, boxes_detection, overlap_metric=sv.OverlapMetric.IOS
        ... )
        array([[0.25, 0.  ],
               [0.  , 0.64]], dtype=float32)

        ```
    """
    #overlap_metric = OverlapMetric.from_value(overlap_metric)
    x_min_true, y_min_true, z_min_true, x_max_true, y_max_true, z_max_true = boxes_true.T
    x_min_det, y_min_det, z_min_det, x_max_det, y_max_det, z_max_det = boxes_detection.T
    count_true, count_det = boxes_true.shape[0], boxes_detection.shape[0]

    if count_true == 0 or count_det == 0:
        return cast(
            np.typing.NDArray[np.float32], np.empty((count_true, count_det), dtype=np.float32)
        )

    x_min_inter = np.empty((count_true, count_det), dtype=np.float32)
    x_max_inter = np.empty_like(x_min_inter)
    y_min_inter = np.empty_like(x_min_inter)
    y_max_inter = np.empty_like(x_min_inter)
    z_min_inter = np.empty_like(x_min_inter)
    z_max_inter = np.empty_like(x_min_inter)

    np.maximum(x_min_true[:, None], x_min_det[None, :], out=x_min_inter)
    np.minimum(x_max_true[:, None], x_max_det[None, :], out=x_max_inter)
    np.maximum(y_min_true[:, None], y_min_det[None, :], out=y_min_inter)
    np.minimum(y_max_true[:, None], y_max_det[None, :], out=y_max_inter)
    np.maximum(z_min_true[:, None], z_min_det[None, :], out=z_min_inter)
    np.minimum(z_max_true[:, None], z_max_det[None, :], out=z_max_inter)

    # we reuse x_max_inter and y_max_inter to store inter_w, inter_h and inter_d
    np.subtract(x_max_inter, x_min_inter, out=x_max_inter)  # inter_w
    np.subtract(y_max_inter, y_min_inter, out=y_max_inter)  # inter_h
    np.subtract(z_max_inter, z_min_inter, out=z_max_inter)  # inter_d
    np.clip(x_max_inter, 0.0, None, out=x_max_inter)
    np.clip(y_max_inter, 0.0, None, out=y_max_inter)
    np.clip(z_max_inter, 0.0, None, out=z_max_inter)

    area_inter = x_max_inter * y_max_inter * z_max_inter # inter_w * inter_h * inter_d

    area_true = (x_max_true - x_min_true) * (y_max_true - y_min_true) * (z_max_true - z_min_true)
    area_det = (x_max_det - x_min_det) * (y_max_det - y_min_det)  * (z_max_det - z_min_det)

    if overlap_metric == "IOU":
        area_norm = area_true[:, None] + area_det[None, :] - area_inter
    elif overlap_metric == "IOS":
        area_norm = np.minimum(area_true[:, None], area_det[None, :])
    else:
        raise ValueError(
            f"overlap_metric {overlap_metric} is not supported, "
            "only 'IOU' and 'IOS' are supported"
        )

    out: np.ndarray[np.float32] = np.zeros_like(area_inter, dtype=np.float32)
    np.divide(area_inter, area_norm, out=out, where=area_norm > 0)
    return out

In [ ]:
def ac_3d(img, init=None, evolve_init=False, dtype="auto", out_max=1, points=100, margin=5, **kwargs):
    if img.ndim == 2:
        img = np.expand_dims(img, axis=0) 

    shape_2d = img.shape[-2:]
    if init is None:
        logger.info(f"creating initial rectangular snake with {points} points and a margin of {margin}")
        r, c = shape_2d
        init = np.array([
            np.concatenate([np.linspace(margin, c-2*margin, points), np.full(points, c-margin), 
                            np.linspace(c-2*margin, margin, points), np.full(points, margin)]),
            np.concatenate([np.full(points, margin), np.linspace(margin, r-2*margin, points), 
                            np.full(points, r-margin), np.linspace(r-2*margin, margin, points)])
        ]).T
    logger.debug(f"img.dtype={img.dtype}, img.shape={img.shape}, init.shape={init.shape}")
    if dtype == "auto":
        dtype = img.dtype
    if evolve_init:
        isnake_mask = np.zeros(img.shape, dtype=dtype)
        isnake_points = np.zeros((img.shape[0], init.shape[0], init.shape[1],), dtype="float64")
        for z in reversed(range(img.shape[0])):
            isnake = segmentation.active_contour(img[z], init, **kwargs)
            isnake_mask[z] = draw.polygon2mask(shape_2d, isnake).astype(dtype)*out_max
            init = isnake
            isnake_points[z] = isnake
    else:
        isnake_points = np.array(Parallel(n_jobs=-1, verbose=10)(delayed(segmentation.active_contour)(i, init, **kwargs) for i in img))
        isnake_mask = np.array(Parallel(n_jobs=-1, verbose=10)(delayed(draw.polygon2mask)(shape_2d, isnake) for isnake in isnake_points))
        isnake_mask = isnake_mask.astype(dtype)*out_max
    return isnake_mask.squeeze(), isnake_points.squeeze()


In [ ]:
def labels_to_masks(labels):
    label_values = (v for v in np.unique(labels) if v > 0)
    masks = []
    for value in label_values:
        mask = labels == value
        masks.append(mask)
    return np.array(masks)

In [ ]:
def group_bboxes(bboxes, divisor=1, threshold=0.5):
    
    def in_groups(item, groups):
        for g in groups:
            if item in g:
                return True
        return False
    
    xyxy = np.mod(bboxes, divisor)
    #print (xyxy[:7])
    if len(bboxes[0]) == 4:
        iou_matrix = sv.box_iou_batch(xyxy, xyxy, overlap_metric=sv.OverlapMetric.IOU)
    elif len(bboxes[0]) == 6:
        iou_matrix = box_iou_batch_3d(xyxy, xyxy, overlap_metric="IOU")
    #print (iou_matrix)
    iou_matrix = np.triu(iou_matrix, k=1)
    pairs = np.argwhere(iou_matrix > threshold)
    
    groups = []
    for i, pair in enumerate(pairs):
        p0 = pair[0]
        #print (i, p0, p1, iou_matrix[p0, p1])
        if not in_groups(p0, groups):
            pairs_with_p0 = np.unique(np.array([p for p in pairs if p[0] == p0]).flatten())
            logger.info(f"Creating new group with {pairs_with_p0}")
            groups.append(pairs_with_p0)
    return groups

In [ ]:
def label_grouped_mask(mask, groups:list[np.ndarray], threshold=1):
    labels = []
    th = math.prod(tile_factor)//2
    for label_value, g in enumerate(groups, 1):
        label_array = (mask[g].sum(axis=0)>threshold)*label_value
        labels.append(label_array)
    labels = np.sum(np.array(labels), axis=0).astype("uint16")
    return labels

# Load Image

In [ ]:
#path="/standard/vol191/siegristlab/Microsam_Segmentation/Conditional Split/control_24+48/DCP1/1_Dpn.tif"
path="/standard/vol191/siegristlab/Microsam_Segmentation/24h/AkhGal4 x OR Susie/Scrib488 Dpn555 EdU 647/Raw files/Animal 1.lif"

scene_index = 0

embedding_path = "/standard/vol191/siegristlab/Sagar/microsam/embeddings/"

In [ ]:
ilc = ImageLoaderConfig(
    squeeze=True, 
    rename_channel={"Red": "Dpn", "Green": "Scrib", "Blue": "EdU"}, 
    scene_index=scene_index, 
    split_channels=True, 
    #substack="C:1"
)
img, metadata = ImageLoader(ilc).run(path)
metadata

# Preprocess

In [ ]:
# Rescale intensity for each channel
scfg = RescaleConfig(
    low=2, 
    high=98, 
    dtype=np.uint8, 
    iterator_config=ArrayIteratorConfig(slice_def=(-3,-2,-1))
)
simg,_ = Rescale(scfg).run(img, metadata=metadata, verbose=1)

In [ ]:
# Apply gaussian blur, separately for each channel and focal plane
gcfg = FuncProcessorConfig(
    func=gaussian,
    kwargs={"sigma": 1.0},
    iterator_config=ArrayIteratorConfig(slice_def=(-2,-1))
)
nimg, _ = FuncProcessor(gcfg).run(simg, verbose=1)    

In [ ]:
gamma = 0.2
enimg = np.array([rescale_intensity(adjust_sigmoid(adjust_gamma(i, gamma=gamma)),out_range="uint8") for i in nimg])
print (np.max(enimg))
#emimg = exposure.rescale_intensity(filters.gaussian(exposure.adjust_sigmoid(exposure.adjust_gamma(mimg, gamma=gamma)), sigma=5.0), out_range="uint8")

In [ ]:
stackview.slice(np.concatenate([simg, rescale_intensity(nimg, out_range="uint8"), enimg], axis=-1))

In [ ]:
# Project all channels to one.

pcfg = FuncProcessorConfig(
    func=np.max, 
    kwargs={"axis":("C")}, 
    strict_axis=False,
    dtype=np.uint16,
)
c_img, c_metadata = FuncProcessor(pcfg).run(enimg, metadata=metadata)
metadata, c_metadata

In [ ]:
stackview.slice(c_img, continuous_update=True)

# Rough Mask of Projection to inform Lobe Segmentation

In [ ]:
#c_mask, c_points = ac_3d(proj, points=100, margin=0, out_max=255, alpha=0.015, beta=0.1, gamma=0.001, w_edge=1.75, boundary_condition="periodic") # beta=0.1, w_edge=1.75
#logging.info(f"{proj.dtype}, {c_mask.dtype}, {np.max(proj)}, {np.max(c_mask)}")

In [ ]:
#stackview.slice(np.concatenate([proj, animg[40], c_mask, (proj-0.2*c_mask)], axis=-1))

In [ ]:
#isnake_img,_ = ac_3d(animg[::4], init=c_points, out_max=255, alpha=0.015, beta=0.1, gamma=0.001, w_edge=1.75, boundary_condition="periodic") #w_edge=1.75

In [ ]:
#stackview.slice(np.concatenate([img[0,::4]+0.2*isnake_img, animg[::4]+0.2*isnake_img, isnake_img], axis=-1))

# Detect Tissue boundaries with MicroSAM (resampling)

## Resize

In [ ]:
factor = 3
width = c_img.shape[-1]//factor
padding = 10

rcfg = ResizeConfig(width=width)
r_img, r_metadata = Resize(rcfg).run(c_img, metadata=c_metadata, verbose=1)
c_metadata, r_metadata

In [ ]:
stackview.orthogonal(r_img)

## Z-Project

In [ ]:
fcfg = FuncProcessorConfig(
    func=np.mean, 
    kwargs={"axis":("Z")},
)
proj, p_metadata = FuncProcessor(fcfg).run(r_img, metadata=r_metadata)

In [ ]:
tile_factor = (factor, factor)
tcfg = TilerConfig(factor=tile_factor, alt_flip=False, pad_width={-2:(0,padding),-1:(0,padding)})
t_proj,t_metadata = Tiler(tcfg).run(proj, metadata=p_metadata)

In [ ]:
rcfg = RegionFilterConfig(filters=[
    RangeFilter(
        RangeFilterConfig(
            attribute="circularity", range=(0.5,1.0)
        ),
    ),
    RangeFilter(
        RangeFilterConfig(
            attribute="aspect_ratio", range=(0.5,1.0)
        ),
    ),
    RangeFilter(
        RangeFilterConfig(
            attribute="cross_sectional_area", range=(1500,np.inf)
        ),
    )]
)
rf = RegionFilter(rcfg)

mcfg = MicroSAMSegmenterConfig(
    iterator_config=ArrayIteratorConfig(slice_def=()),
    embedding_path=embedding_path,
)
microsam = MicroSAMSegmenter(mcfg)

scfg = SegmentationFlowConfig(
    segmenter = microsam,
    region_filter = rf,
)

p_labels = SegmentationFlow(scfg).run(t_proj, metadata=t_metadata)

In [ ]:
racfg = RegionAnalyzerConfig(
    properties=["area", "cross_sectional_area", "bbox", "aspect_ratio", "circularity", "perimeter"],
    iterator_config = ArrayIteratorConfig(slice_def=()),
    output_type="dataframe"
)
ra = RegionAnalyzer(racfg)

p_results = ra.run(p_labels, metadata=t_metadata)

In [ ]:
p_results.describe()

In [ ]:
stackview.blend(t_proj, p_labels, blend_factor=40, continuous_update=True)

# Consensus voting
Find matching regions based on IoU of bounding boxes, then create consensus masks (only consider pixels that show up in at least half of the masks).

In [ ]:
logger.info(f"width={width}, p_labels.shape={p_labels.shape}, {p_labels.shape[-1]//factor}")
groups = group_bboxes(p_results[["bbox-1", "bbox-0", "bbox-3","bbox-2"]].to_numpy()-np.array((0,0,1,1)), divisor=p_labels.shape[-1]//factor, threshold=0.5)

ucfg = UntilerConfig(
    factor = tile_factor,
    iterator_config = ArrayIteratorConfig(slice_def=())
)

masks = labels_to_masks(p_labels)
untiled,_ = Untiler(ucfg).run(masks)
#untiled = untile(masks, tile_factor)
u_proj =  np.sum(untiled>0, axis=0)>0

labels = label_grouped_mask(u_proj, groups, threshold=math.prod(tile_factor)//2)
labels = labels[..., 0:labels.shape[-2]-padding, 0:labels.shape[-1]-padding]
stackview.blend(proj, labels, blend_factor=25, continuous_update=True)

# Segment tiled Z-stack

In [ ]:
tr_img,tr_metadata = Tiler(tcfg).run(r_img, metadata=r_metadata)

In [ ]:
mcfg = MicroSAMSegmenterConfig(
    iterator_config=ArrayIteratorConfig(slice_def=()),
    embedding_path=embedding_path,
)
microsam = MicroSAMSegmenter(mcfg)

scfg = SegmentationFlowConfig(
    segmenter = microsam,
    region_filter = RegionFilter(
        RegionFilterConfig(
            filters=[
                RangeFilter(
                    RangeFilterConfig(
                        attribute="cross_sectional_area", 
                        range=(3000, np.inf)
                    )
                ),
                RangeFilter(
                    RangeFilterConfig(
                        attribute="aspect_ratio", 
                        range=(0.5, 1.0)
                    )
                ),
            ]
        )
    )
)

tlabels = SegmentationFlow(scfg).run(tr_img, metadata=tr_metadata)

In [ ]:
ra=RegionAnalyzer(
   RegionAnalyzerConfig(
        output_type="dataframe", 
        properties=["cross_sectional_area", "aspect_ratio", "bbox", "volume", "aspect_ratio"],
        iterator_config=ArrayIteratorConfig(slice_def=(-3,-2,-1))
    )
)
tresults = ra.run(tlabels, metadata=tr_metadata)

In [ ]:
stackview.blend(tr_img, tlabels, blend_factor=25, continuous_update=True)

In [ ]:
tresults.describe()

In [ ]:
tgroups = group_bboxes(tresults[["bbox-2", "bbox-1", "bbox-0", "bbox-5", "bbox-4", "bbox-3"]].to_numpy()-np.array((0,0,0,1,1,1)), divisor=tlabels.shape[-1]//factor, threshold=0.5)

tmasks = labels_to_masks(tlabels)

ucfg = UntilerConfig(
    factor = tile_factor,
    iterator_config = ArrayIteratorConfig(slice_def=())
)
untiled,_ = Untiler(ucfg).run(tmasks)
tproj =  np.sum(untiled>0, axis=0)>0

labels = label_grouped_mask(tproj, tgroups, threshold=math.prod(tile_factor)//2)

# remove padding on bottom and right edge
cropped_height = labels.shape[-2]-padding
cropped_width = labels.shape[-1]-padding
cropped_labels = labels[..., 0:cropped_height, 0:cropped_width]

In [ ]:
ecfg = ResizeConfig(width=img.shape[-1], normalize=False, dtype=np.uint16)
labels, l_metadata = Resize(ecfg).run(cropped_labels, metadata=r_metadata)

stackview.blend(
    c_img.astype(np.uint16),
    labels.astype(np.uint64), 
    blend_factor=40 # Sets the transparency of the overlay (0.0 to 1.0)
)

# Save label

In [ ]:
c_metadata["channel_names"] = "Merged-" + "-".join(c_metadata["channel_names"])

In [ ]:
from vistiq.io import ImageWriterConfig, ImageWriter
imc = ImageWriterConfig()
outpath = ".".join(path.split(".")[:-1]) + "-lobes.tif"
ImageWriter(imc).run(labels, outpath, metadata=c_metadata)